In [9]:
#RAG stands for Retrieval-Augmented Generation.
# Retrieval-Augmented Generation (RAG) is the process of optimizing the
#  output of a large language model, so it references an authoritative 
# knowledge base outside of its training data sources before generating a 
# response. Large Language Models (LLMs) are trained on vast volumes of data
#  and use billions of parameters to generate original output for tasks like
#  answering questions, translating languages, and completing sentences.
#  RAG extends the already powerful capabilities of LLMs to specific domains 
# or an organization's internal knowledge base, all without the need to retrain
#  the model. It is a cost-effective approach to improving LLM output so it 
# remains relevant, accurate, and useful in various contexts. 
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np


c:\Users\rawat\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# PART 1: Embeddings — Understanding them deeply

In [10]:
print("=" * 55)
print("PART 1: Embeddings")
print("=" * 55)

# Load a free embedding model
# all-MiniLM-L6-v2 is the most popular beginner model
# - Runs on your laptop (no API key needed)
# - Converts any text to 384 numbers
# - Downloads automatically first time (~80MB)
print("\nLoading embedding model (first time takes ~1 min)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded!")





PART 1: Embeddings

Loading embedding model (first time takes ~1 min)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3845.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!


# SECTION 1.1: Convert text to vectors

In [ ]:
print("\n--- Converting text to vectors ---")

sentence="The patient has a sever headache"
embedding =embedder.encode(sentence)#convert sentence to numerical vector

print(f"Sentence: {sentence}")
print(f"Vector shape: {embedding.shape}")#(384,)
print(f"First 10 numbers :{embedding[:10]}")
print(f"These 384 numbers represent the MEANING of the sentence")




--- Converting text to vectors ---
[-3.37630697e-02  3.56581733e-02 -5.90981506e-02  1.71655836e-03
 -9.22339931e-02 -1.70113519e-02  4.44667339e-02  6.51354864e-02
  9.24691744e-03 -3.38907391e-02  1.61484499e-02 -5.28666824e-02
 -1.22835115e-02  2.54537910e-02 -7.36460239e-02 -1.39070367e-02
  2.02620663e-02 -5.47414161e-02  6.34525642e-02  5.46432026e-02
 -1.07168086e-01  4.75777350e-02 -3.00781578e-02  3.20506580e-02
 -9.48108360e-02  3.22380662e-02  2.99291071e-02 -7.99535811e-02
  4.02597673e-02  4.84080706e-03  2.50477735e-02 -4.41421270e-02
 -7.51447901e-02 -2.66016349e-02  3.91830616e-02  1.05109354e-02
 -6.84655532e-02  5.41264303e-02 -4.07610722e-02  1.05299890e-01
 -4.84290384e-02 -2.25372054e-02 -1.18554771e-01 -1.82652641e-02
  2.22386755e-02 -2.34147105e-02 -1.32803796e-02  3.97331594e-03
  7.35498220e-02  4.96258810e-02 -6.57080635e-02 -7.44513348e-02
 -1.33844733e-04  9.23778955e-03 -1.87535472e-02 -2.99201570e-02
 -7.47952098e-03  4.23434982e-03 -6.04962073e-02  4.59

# SECTION 1.2: Similarity — the core concept

In [24]:
print("\n--- Similarity between sentences ---")

def cosine_similarity(a,b):
    """
    Measures how similar two vectors are.
    1.0 = identical meaning
    0.0 = completely unrelated
    Formula: dot product / (magnitude of a * magnitude of b)
    """
    #linalg->linear algebra norm->normalized form
    return float(np.dot(a,b)/(np.linalg.norm(a) * np.linalg.norm(b)) )

sentences = [
    "The patient has a severe headache",       # base sentence
    "The person is experiencing head pain",    # similar meaning
    "The doctor prescribed medication",        # related topic
    "I love eating pizza on weekends",         # completely unrelated
    "Machine learning is fascinating",         # unrelated
]
base =embedder.encode(sentences[0])#to check similarity with other sentences
others=embedder.encode(sentences[1:])

print(f"\nBase: '{sentences[0]}'")
print("-" * 50)

# zip->Combines:
# sentence text
# its embedding ("I like dogs", [0.1, 0.2, ...])
# enumerate(...)
# Adds an index (i)
# Useful for numbering (not used much here)

#sent->sentences[1:]
#emb = embedding (vector of numbers) for that sentence
for i,(sent,emb) in enumerate(zip(sentences[1:],others)):
    sim=cosine_similarity(base,emb)
    bar = "█" * int(sim * 20)
    print(f"{sim:.2f} {bar}")
    print(f"     '{sent}'")

# 🔑 KEY INSIGHT:
# "headache" vs "head pain" → ~0.85 (very similar)
# "headache" vs "pizza"     → ~0.10 (unrelated)
# This is WHY embeddings power RAG — semantic understanding



--- Similarity between sentences ---

Base: 'The patient has a severe headache'
--------------------------------------------------
0.73 ██████████████
     'The person is experiencing head pain'
0.45 ████████
     'The doctor prescribed medication'
-0.01 
     'I love eating pizza on weekends'
-0.04 
     'Machine learning is fascinating'


# SECTION 1.3: Sentence embeddings (whole sentences)


In [25]:
print("\n--- Whole sentence similarity ---")

questions = [
    "What are the side effects of the medicine?",
    "What adverse reactions does the drug have?",    # same meaning, different words
    "How do I cook pasta?",                          # unrelated
]

q_embeddings=embedder.encode(questions)
base_q=q_embeddings[0]
print(f"Base question: '{questions[0]}'")
print()
for i in range(1,len(questions)):
    sim=cosine_similarity(base_q,q_embeddings[i])
    print(f"Similarity: {sim:.2f} → '{questions[i]}'")

# 🔑 "side effects" vs "adverse reactions" → ~0.80
# This is why RAG finds the right chunk even when
# the user's question uses different words than the document




--- Whole sentence similarity ---
Base question: 'What are the side effects of the medicine?'

Similarity: 0.76 → 'What adverse reactions does the drug have?'
Similarity: 0.10 → 'How do I cook pasta?'


# PART 2: ChromaDB — Your Vector Database

In [26]:
#It stores embeddings (vectors of numbers) and helps you search by meaning, not just keywords.
#What ChromaDB does Stores embeddings "Hello world" → [0.12, -0.45, ...]
#Lets you search by meaning-
# Query: "animals"
# ChromaDB can return:
# "I love cats"
# "Dogs are friendly"
# 👉 Even if the word "animals" isn’t present

In [30]:
print("\n" + "=" * 55)
print("PART 2: ChromaDB")
print("=" * 55)

# Initialize ChromaDB — runs entirely on your laptop
# No signup, no API key, free forever
client=chromadb.Client()

# Create a collection (like a table in MongoDB)
# Each collection stores embeddings + their text + metadata
collection = client.get_or_create_collection(
    name ="my_first_collection",
    metadata={"description": "learning ChromaDB"}
)
print("✅ ChromaDB collection created!")





PART 2: ChromaDB
✅ ChromaDB collection created!


# SECTION 2.1: Adding documents


In [ ]:
print("\n--- Adding documents to ChromaDB ---")

# Sample text chunks (pretend these came from a PDF)
documents = [
    "Python is a high-level programming language known for its simplicity.",
    "Machine learning is a subset of AI that learns from data.",
    "Neural networks are inspired by the human brain structure.",
    "Flask is a lightweight Python web framework for building APIs.",
    "ChromaDB is a vector database that stores text embeddings.",
    "React is a JavaScript library for building user interfaces.",
    "MongoDB is a NoSQL document database used in web development.",
    "Sentiment analysis is the process of identifying emotion in text.",
    "TF-IDF converts text to numbers based on word frequency.",
    "Embeddings capture semantic meaning of text as dense vectors.",
]

# Metadata — store extra info like page numbers, source file
metadatas = [
    {"topic": "python",    "page": 1},
    {"topic": "ml",        "page": 2},
    {"topic": "ml",        "page": 3},
    {"topic": "flask",     "page": 4},
    {"topic": "chromadb",  "page": 5},
    {"topic": "react",     "page": 6},
    {"topic": "mongodb",   "page": 7},
    {"topic": "nlp",       "page": 8},
    {"topic": "nlp",       "page": 9},
    {"topic": "embeddings","page": 10},
]
# Generate embeddings for all documents
print("Generating embeddings...")
embeddings = embedder.encode(documents).tolist()
#Add to ChromaDB
collection.add(
    documents=documents,#optional documents for each record
    embeddings=embeddings,
    metadatas=metadatas,
    ids= [str(i) for i in range(len(documents))]#record ids to add
)
print(f"✅ Added {len(documents)} documents to ChromaDB!")
print(f"Collection count: {collection.count()}")





--- Adding documents to ChromaDB ---
Generating embeddings...
[[-0.060378286987543106, 0.02286936156451702, -0.018337979912757874, 0.017995942384004593, -0.032349180430173874, -0.15243403613567352, -0.0256672203540802, 0.0894085019826889, -0.08026307076215744, 0.00828357320278883, -0.050127558410167694, 0.04592454805970192, 0.0730910673737526, 0.023825814947485924, 0.066657155752182, -0.051384106278419495, -0.05324687063694, -0.004387068096548319, -0.00099912378937006, -0.1373683512210846, -0.052452731877565384, 0.04521123319864273, -0.02272624894976616, 0.008147669956088066, 0.019512861967086792, 0.0077009499073028564, -0.012988265603780746, -0.0169870313256979, 0.038005530834198, -0.0056867459788918495, -0.043503668159246445, 0.09614720195531845, 0.0734669640660286, 0.018171900883316994, 0.009827056899666786, 0.07490966469049454, 0.0046404278837144375, -0.08146166801452637, -0.0542481504380703, 0.012999646365642548, -0.07108495384454727, 0.04457436501979828, -0.061756350100040436, -

# SECTION 2.2: Querying — semantic search

In [35]:
print("\n--- Semantic Search ---")

#Take a query → convert to embedding → search ChromaDB → print results”
def search(query,n_results=3):
    query_embedding=embedder.encode([query]).tolist()#Because most embedding models expect:["text1", "text2"]
    #Some models return NumPy arrays

    results=collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]#what data to return
    )

    print(f"\nQuery: '{query}'")
    print(f"Top {n_results} results:")
    print("-" * 50)

    #{
    #"documents": [[doc1, doc2, doc3]],
    # }
    #  👉 Outer list = batch
    # 👉 Inner list = actual results
    for i in range(len(results["documents"][0])):
        doc=results["documents"][0][i]
        meta=results["metadatas"][0][i]
        distance = results["distances"][0][i]
        similarity=1-distance
        print(f"Result {i+1} (similarity: {similarity:.2f})")
        print(f"  Page {meta['page']}: {doc}")
    print()

# Test different queries
search("How do neural networks work?")
search("What is used for building web APIs?")
search("How does text get converted to numbers?")

# 🔑 Notice: "web APIs" finds Flask even though query didn't say Flask
# That's semantic search — understanding MEANING not just keywords



--- Semantic Search ---

Query: 'How do neural networks work?'
Top 3 results:
--------------------------------------------------
Result 1 (similarity: 0.28)
  Page 3: Neural networks are inspired by the human brain structure.
Result 2 (similarity: -0.21)
  Page 2: Machine learning is a subset of AI that learns from data.
Result 3 (similarity: -0.61)
  Page 6: React is a JavaScript library for building user interfaces.


Query: 'What is used for building web APIs?'
Top 3 results:
--------------------------------------------------
Result 1 (similarity: 0.21)
  Page 4: Flask is a lightweight Python web framework for building APIs.
Result 2 (similarity: -0.06)
  Page 6: React is a JavaScript library for building user interfaces.
Result 3 (similarity: -0.30)
  Page 1: Python is a high-level programming language known for its simplicity.


Query: 'How does text get converted to numbers?'
Top 3 results:
--------------------------------------------------
Result 1 (similarity: 0.06)
  Page 9: 

# SECTION 2.3: Persistent storage


In [ ]:
print("--- Persistent ChromaDB (saves to disk) ---")
# The above used in-memory ChromaDB (resets when Python restarts)
# For your RAG app you need persistent storage (saves to disk)
import os

persistent_client=chromadb.PersistentClient(path="./chroma_db")

try:
    persistent_collection=persistent_client.create_collection("rag_documents")
    print("✅ Created new persistent collection")
except: 
    persistent_collection = persistent_client.get_collection("rag_documents")
    print("✅ Loaded existing persistent collection")

persistent_collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=[str(i) for i in range(len(documents))]
)

print(f"Saved {persistent_collection.count()} documents to ./chroma_db/")
print("These persist even after Python restarts!")




# PART 3: Complete Mini RAG Pipeline 

In [43]:
# Just retrieval — we add the LLM answer on Day 2

print("\n" + "=" * 55)
print("PART 3: Mini RAG Pipeline")
print("=" * 55)

def mini_rag(question,n_chunks=3):
    """
    Simple RAG pipeline:
    1. Convert question to embedding
    2. Find most relevant chunks
    3. Return chunks as context (LLM added Day 2)
    """
    print(f"\nQuestions: '{question}'")
    print("=" * 50)
    q_embedding=embedder.encode([question]).tolist()#Turns the question into a vector (numbers)
    results=collection.query(
        #Compares your question vector with all stored document vectors
        # Finds the top 3 closest (most similar meaning)
        query_embeddings=q_embedding,
        n_results=n_chunks,
        include=["documents","metadatas","distances"]
    )
    print(results)
    # Step 3: Build context
    print(f"Found {n_chunks} relevant chunks:")
    context_parts=[]
    for i in range(len(results["documents"][0])):
        doc=results['documents'][0][i]#Gets the actual sentences from your dataset
        meta     = results["metadatas"][0][i]
        distance = results["distances"][0][i]
        similarity =round(1-distance,2)
        print(f"\n  Chunk {i+1} — Page {meta['page']} (relevance: {similarity})")
        print(f"  {doc}")
        context_parts.append(f"[Page {meta['page']}]:{doc}")

    context = "\n".join(context_parts)
    print(f"\nContext that would be sent to LLM:")
    print("-" * 50)
    print(context)
    print("-" * 50)
    print("(On Day 2 we send this to Groq LLM for the final answer)")
    return context

mini_rag("What tools are used for machine learning?")
mini_rag("How does text get understood by computers?")



PART 3: Mini RAG Pipeline

Questions: 'What tools are used for machine learning?'
{'ids': [['1', '0', '4']], 'embeddings': None, 'documents': [['Machine learning is a subset of AI that learns from data.', 'Python is a high-level programming language known for its simplicity.', 'ChromaDB is a vector database that stores text embeddings.']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'topic': 'ml', 'page': 2}, {'page': 1, 'topic': 'python'}, {'topic': 'chromadb', 'page': 5}]], 'distances': [[0.8704198002815247, 1.406233549118042, 1.4486708641052246]]}
Found 3 relevant chunks:

  Chunk 1 — Page 2 (relevance: 0.13)
  Machine learning is a subset of AI that learns from data.

  Chunk 2 — Page 1 (relevance: -0.41)
  Python is a high-level programming language known for its simplicity.

  Chunk 3 — Page 5 (relevance: -0.45)
  ChromaDB is a vector database that stores text embeddings.

Context that would be sent to LLM:
-------------------

'[Page 10]:Embeddings capture semantic meaning of text as dense vectors.\n[Page 8]:Sentiment analysis is the process of identifying emotion in text.\n[Page 9]:TF-IDF converts text to numbers based on word frequency.'